# ASReview 5-Star: Getting Started

This notebook introduces the ASReview 5-Star package for enhanced systematic review workflows.

## What You'll Learn
- Installing and importing ASReview 5-Star
- Basic stopping rule calculations
- Inter-rater reliability (IRR) metrics
- Meta-analysis basics
- PRISMA flow diagram statistics

## Installation

Install ASReview 5-Star with pip:

In [ ]:
# Install the package (uncomment to run)
# !pip install -e ../

# Or install with all extras
# !pip install -e ../[all]

## Import the Package

In [ ]:
import asreview_5star as a5s
import math

print(f"ASReview 5-Star version: {a5s.__version__}")

## 1. Stopping Rules

Stopping rules help determine when you can safely stop screening in a systematic review while maintaining high recall.

### Bayesian Stopping Rule

Uses a Beta-Binomial model to estimate the probability that target recall has been achieved.

In [ ]:
# Example: You've screened 500 out of 5000 records and found 25 relevant
result = a5s.bayesian_stopping(
    n_screened=500,
    n_relevant=25,
    n_total=5000,
    target_recall=0.95
)

print(f"Should stop: {result.should_stop}")
print(f"Confidence: {result.confidence:.2%}")
print(f"Message: {result.message}")
print(f"\nDetails:")
for key, value in result.details.items():
    print(f"  {key}: {value}")

### SPRT (Sequential Probability Ratio Test)

Tests whether the prevalence is below a threshold, indicating it's safe to stop.

In [ ]:
result = a5s.sprt_stopping(
    n_screened=500,
    n_relevant=10,
    null_prevalence=0.01,
    alt_prevalence=0.05
)

print(f"Should stop: {result.should_stop}")
print(f"Decision: {result.details.get('decision', 'continue')}")
print(f"Message: {result.message}")

## 2. Inter-Rater Reliability (IRR)

IRR metrics measure agreement between screeners.

### Cohen's Kappa

Measures agreement between two raters, accounting for chance agreement.

In [ ]:
# Two screeners rated 20 records (0=exclude, 1=include)
screener1 = [1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0]
screener2 = [1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0]

result = a5s.cohens_kappa(screener1, screener2)

print(f"Cohen's Kappa: {result.coefficient:.4f}")
print(f"95% CI: [{result.ci_lower:.4f}, {result.ci_upper:.4f}]")
print(f"Interpretation: {result.interpretation}")

### Fleiss' Kappa

For multiple raters (3+ screeners).

In [ ]:
# 10 records rated by 3 screeners
ratings = [
    [1, 1, 1],  # Record 1: all agree include
    [0, 0, 0],  # Record 2: all agree exclude
    [1, 1, 0],  # Record 3: 2 include, 1 exclude
    [0, 0, 1],  # Record 4: 2 exclude, 1 include
    [1, 1, 1],
    [0, 1, 0],
    [1, 0, 1],
    [0, 0, 0],
    [1, 1, 1],
    [0, 0, 0]
]

result = a5s.fleiss_kappa(ratings)

print(f"Fleiss' Kappa: {result.coefficient:.4f}")
print(f"Number of raters: {result.details['n_raters']}")
print(f"Interpretation: {result.interpretation}")

## 3. Meta-Analysis

Pool effect sizes from multiple studies.

### Example: Pooling Hazard Ratios

In [ ]:
# Study data (log hazard ratios and standard errors)
studies = [
    {"name": "Smith 2020", "hr": 0.80, "ci_lower": 0.65, "ci_upper": 0.98},
    {"name": "Jones 2021", "hr": 0.75, "ci_lower": 0.60, "ci_upper": 0.94},
    {"name": "Brown 2022", "hr": 0.85, "ci_lower": 0.70, "ci_upper": 1.03},
    {"name": "Davis 2023", "hr": 0.70, "ci_lower": 0.55, "ci_upper": 0.89},
]

# Convert to log scale
effects = [math.log(s["hr"]) for s in studies]
ses = [(math.log(s["ci_upper"]) - math.log(s["ci_lower"])) / 3.92 for s in studies]
names = [s["name"] for s in studies]

# Pool with random effects model
result = a5s.pool_effects(effects, ses, study_names=names, model="random")

print(f"Model: {result.model}")
print(f"\nPooled Effect (log HR): {result.pooled_effect:.4f}")
print(f"Pooled HR: {math.exp(result.pooled_effect):.3f}")
print(f"95% CI: [{math.exp(result.ci_lower):.3f}, {math.exp(result.ci_upper):.3f}]")
print(f"P-value: {result.p_value:.4f}")
print(f"\nHeterogeneity:")
print(f"  I-squared: {result.heterogeneity['I_squared']:.1f}%")
print(f"  Tau-squared: {result.heterogeneity['tau_squared']:.4f}")

### Publication Bias Tests

In [ ]:
# Egger's test
egger = a5s.eggers_test(effects, ses)
print(f"Egger's Test:")
print(f"  P-value: {egger.p_value:.4f}")
print(f"  Interpretation: {egger.interpretation}")

# Begg's test
begg = a5s.beggs_test(effects, ses)
print(f"\nBegg's Test:")
print(f"  P-value: {begg.p_value:.4f}")
print(f"  Interpretation: {begg.interpretation}")

## 4. PRISMA Statistics

Calculate PRISMA 2020 flow diagram statistics.

In [ ]:
from asreview_5star import prisma_stats, prisma_flow_data

stats = prisma_stats(
    records_total=3500,
    records_duplicates=500,
    records_screened=3000,
    records_excluded_screening=2700,
    records_retrieved=300,
    records_not_retrieved=15,
    records_excluded_fulltext=260,
    exclusion_reasons={
        "Wrong population": 80,
        "Wrong intervention": 70,
        "Wrong outcome": 50,
        "Wrong study design": 40,
        "Conference abstract only": 20
    }
)

print(f"PRISMA Flow Statistics:")
print(f"  Records identified: {stats.records_identified_databases}")
print(f"  Duplicates removed: {stats.records_removed_duplicates}")
print(f"  Records screened: {stats.records_screened}")
print(f"  Records excluded (screening): {stats.records_excluded_screening}")
print(f"  Full-texts assessed: {stats.reports_assessed}")
print(f"  Studies included: {stats.studies_included}")

In [ ]:
# Get structured flow data
flow = prisma_flow_data(stats)

print(f"\nYield Rate: {flow['summary']['yield_rate']:.2f}%")
print(f"Screening NNR: {flow['summary']['screening_nnr']:.1f}")

## Next Steps

- **02_stopping_rules_comparison.ipynb**: Deep dive into stopping rules
- **03_meta_analysis_workflow.ipynb**: Complete meta-analysis workflow
- **04_audit_certification.ipynb**: Audit trail and certification

## Resources

- [ASReview Documentation](https://asreview.nl/)
- [API Reference](/docs)
- [GitHub Repository](https://github.com/asreview/asreview-5star)